# 08 - Weather + Telegram message

Combines `get_weather` with `send_telegram_message`, using the official [Telegram Bot API](https://core.telegram.org/bots/api) - free, no per-message cost, and reliable from any country (no carrier/phone number involved).

**Setup:**
1. In Telegram, message **@BotFather** and send `/newbot`. Follow the prompts (pick a name and a username ending in `bot`). It replies with a bot token like `123456789:ABCdefGhIJKlmNoPQRstuVwxYZ`.
2. Open a chat with your new bot (search its username) and send it any message, e.g. `hi`. Telegram bots can't message you first - this is what lets the bot know who you are.
3. Enter the bot token below when prompted.

## 1. Install dependencies

In [11]:
%pip install -q anthropic requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Set your credentials

The bot token uses `getpass` - anyone with it can control your bot.

In [12]:
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")
if not os.environ.get("TELEGRAM_BOT_TOKEN"):
    os.environ["TELEGRAM_BOT_TOKEN"] = getpass("Enter your Telegram bot token: ")

# Only needed if you hit: "anthropic-workspace-id is required when
# authenticating with an identity-linked API key" - leave blank to skip.
if not os.environ.get("ANTHROPIC_WORKSPACE_ID"):
    _workspace_id = input("ANTHROPIC_WORKSPACE_ID (leave blank if not needed): ").strip()
    if _workspace_id:
        os.environ["ANTHROPIC_WORKSPACE_ID"] = _workspace_id

## 3. Find your chat_id

Make sure you've already sent your bot a message in the Telegram app (step 2 above), then run this cell - it reads that message back to discover your `chat_id` automatically. If it says no messages were found, send the bot a message now and re-run this cell.

In [20]:
import requests

TELEGRAM_API = f"https://api.telegram.org/bot{os.environ['TELEGRAM_BOT_TOKEN']}"


def discover_chat_id():
    """Look up your chat_id from the bot's recent messages. Telegram bots
    can't message you first - you have to message the bot at least once,
    which is what this is reading back."""
    try:
        resp = requests.get(f"{TELEGRAM_API}/getUpdates", timeout=15)
        resp.raise_for_status()
    except requests.RequestException as exc:
        print(f"Error: could not reach Telegram ({exc}). Check your bot token.")
        return None
    results = resp.json().get("result", [])
    if not results:
        return None
    return results[-1]["message"]["chat"]["id"]


if not os.environ.get("TELEGRAM_CHAT_ID"):
    found = discover_chat_id()
    if found:
        os.environ["TELEGRAM_CHAT_ID"] = str(found)
        print(f"Found chat_id: {found}")
    else:
        print(
            "No messages found yet. In Telegram, open a chat with your bot "
            "and send it anything (e.g. \"hi\"), then re-run this cell."
        )
else:
    print(f"Using existing TELEGRAM_CHAT_ID: {os.environ['TELEGRAM_CHAT_ID']}")


Using existing TELEGRAM_CHAT_ID: 8706613452


## 4. Tools, the agent loop, and the `Agent` class

Same cost cap and turn-collapsing core as the other notebooks. Same "only act when asked" system prompt pattern as the email/notification notebooks.

In [14]:
import datetime
import json
import os

import anthropic

MODEL = "claude-haiku-4-5"
MAX_TOKENS = int(os.environ.get("AGENT_MAX_TOKENS", "1024"))

# claude-haiku-4-5 pricing, $/1M tokens - update if you switch models.
INPUT_COST_PER_MTOK = 1.00
OUTPUT_COST_PER_MTOK = 5.00

# Hard spending cap for this notebook kernel session.
MAX_COST_USD = float(os.environ.get("AGENT_MAX_COST_USD", "0.20"))


class BudgetExceededError(RuntimeError):
    pass


import requests

SYSTEM_PROMPT = (
    "You are a helpful assistant with get_weather and send_telegram_message "
    "tools. Use get_weather for current conditions. Only call "
    "send_telegram_message when the user explicitly asks you to message, "
    "notify, or send something to Telegram - never on your own initiative. "
    "Otherwise reply directly."
)

TOOLS = [
    {
        "name": "get_weather",
        "description": "Get current weather conditions for a location by city name.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name, optionally with country, e.g. 'Paris, France'.",
                },
            },
            "required": ["location"],
        },
    },
    {
        "name": "send_telegram_message",
        "description": "Send a text message to the user via their Telegram bot.",
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {"type": "string", "description": "The message text to send."},
            },
            "required": ["message"],
        },
    },
]

# WMO weather interpretation codes (used by Open-Meteo's weather_code field).
WMO_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    56: "Light freezing drizzle", 57: "Dense freezing drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    66: "Light freezing rain", 67: "Heavy freezing rain",
    71: "Slight snow fall", 73: "Moderate snow fall", 75: "Heavy snow fall",
    77: "Snow grains",
    80: "Slight rain showers", 81: "Moderate rain showers", 82: "Violent rain showers",
    85: "Slight snow showers", 86: "Heavy snow showers",
    95: "Thunderstorm", 96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail",
}


def _request_with_retry(method, url, attempts=2, **kwargs):
    last_exc = None
    for attempt in range(attempts):
        try:
            resp = requests.request(method, url, timeout=15, **kwargs)
            resp.raise_for_status()
            return resp
        except requests.RequestException as exc:
            last_exc = exc
    raise last_exc


def get_weather(location: str) -> str:
    """Free, no-API-key weather lookup via Open-Meteo."""
    try:
        geo_resp = _request_with_retry(
            "GET",
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": location, "count": 1},
        )
        geo_data = geo_resp.json()
    except requests.RequestException as exc:
        return f"Error: location lookup failed ({exc})"

    results = geo_data.get("results")
    if not results:
        return f"Error: could not find a location matching '{location}'."
    place = results[0]

    try:
        weather_resp = _request_with_retry(
            "GET",
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,wind_speed_10m,weather_code",
            },
        )
        weather_data = weather_resp.json()
    except requests.RequestException as exc:
        return f"Error: weather lookup failed ({exc})"

    current = weather_data.get("current", {})
    units = weather_data.get("current_units", {})
    condition = WMO_CODES.get(current.get("weather_code"), "Unknown conditions")
    place_label = place["name"] + ((", " + place["country"]) if place.get("country") else "")

    return (
        "Weather in " + place_label + ": " + condition + ", "
        + str(current.get("temperature_2m")) + units.get("temperature_2m", "\u00b0C")
        + ", wind " + str(current.get("wind_speed_10m")) + " " + units.get("wind_speed_10m", "km/h")
    )


def send_telegram_message(message: str) -> str:
    """Send via the Telegram Bot API - free, no signup beyond creating the
    bot with @BotFather. Uses the TELEGRAM_API base URL and chat_id set up
    in the cell above."""
    chat_id = os.environ.get("TELEGRAM_CHAT_ID")
    if not chat_id:
        return "Error: TELEGRAM_CHAT_ID not set - run the discovery cell above first."
    try:
        _request_with_retry(
            "POST",
            f"{TELEGRAM_API}/sendMessage",
            json={"chat_id": chat_id, "text": message},
        )
        return "Telegram message sent."
    except requests.RequestException as exc:
        return f"Error: failed to send Telegram message ({exc})"


def execute_tool(name: str, tool_input: dict) -> str:
    if name == "get_weather":
        return get_weather(tool_input["location"])
    if name == "send_telegram_message":
        return send_telegram_message(tool_input["message"])
    return f"Error: unknown tool '{name}'"


MAX_PAUSE_RESUMES = 10


def build_client() -> anthropic.Anthropic:
    """Some API keys (personal keys not scoped to one workspace) require an
    anthropic-workspace-id header on every request - see
    https://platform.claude.com/docs/en/manage-claude/authentication#select-a-workspace.
    Set ANTHROPIC_WORKSPACE_ID if you hit: 'anthropic-workspace-id is
    required when authenticating with an identity-linked API key'."""
    workspace_id = os.environ.get("ANTHROPIC_WORKSPACE_ID")
    if workspace_id:
        return anthropic.Anthropic(
            default_headers={"anthropic-workspace-id": workspace_id}
        )
    return anthropic.Anthropic()


class Agent:
    """A minimal conversational agent that can call tools in a loop."""

    def __init__(self, client: anthropic.Anthropic | None = None):
        self.client = client or build_client()
        self.messages: list[dict] = []
        self.total_cost_usd = 0.0

    def send(self, user_input: str) -> str:
        turn_start = len(self.messages)
        self.messages.append({"role": "user", "content": user_input})

        resumes = 0
        while True:
            if self.total_cost_usd >= MAX_COST_USD:
                raise BudgetExceededError(
                    f"Session cost ${self.total_cost_usd:.4f} has reached the "
                    f"${MAX_COST_USD:.4f} cap (AGENT_MAX_COST_USD). Raise the "
                    "cap or start a new session to continue."
                )

            response = self.client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                tools=TOOLS,
                messages=self.messages,
            )
            self.total_cost_usd += (
                response.usage.input_tokens * INPUT_COST_PER_MTOK
                + response.usage.output_tokens * OUTPUT_COST_PER_MTOK
            ) / 1_000_000
            self.messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "pause_turn":
                resumes += 1
                if resumes > MAX_PAUSE_RESUMES:
                    break
                continue

            if response.stop_reason != "tool_use":
                break

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        }
                    )
            self.messages.append({"role": "user", "content": tool_results})

        reply = "".join(
            block.text for block in response.content if block.type == "text"
        )
        self.messages[turn_start:] = [
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": reply},
        ]
        return reply


## 5. Create the agent

In [15]:
agent = Agent()
print(f"Agent ready (cap ${MAX_COST_USD:.4f} for this kernel session)")

Agent ready (cap $0.2000 for this kernel session)


## 6. Try it

In [16]:
reply = agent.send(
    "What's the weather in Tokyo, and send it to me on Telegram."
)
print(reply)
print(f"(session cost so far: ~${agent.total_cost_usd:.4f})")

Done! I've sent the weather information to your Telegram. The weather in Tokyo is currently showing dense drizzle with a temperature of 21.6°C and light winds at 2.6 km/h.
(session cost so far: ~$0.0035)


## 7. Optional: interactive chat loop

Type `exit` to stop.

In [17]:
while True:
    user_input = input("You: ")
    if user_input.strip().lower() in {"exit", "quit"}:
        break
    try:
        reply = agent.send(user_input)
    except BudgetExceededError as exc:
        print(f"Agent: [stopped] {exc}")
        break
    print(f"Agent: {reply}  (session cost so far: ~${agent.total_cost_usd:.4f})")


Agent: Done! I've sent the weather information for Kathmandu to your Telegram. It's currently showing moderate drizzle with a temperature of 20.8°C and winds at 2.5 km/h.  (session cost so far: ~$0.0072)


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.4: user messages must have non-empty content'}, 'request_id': 'req_011CeiYzzn91foTSJiEsiih9'}